<h1>New</h1>

In [1]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Lambda
import tensorflow.keras.backend as K

In [2]:
sentences = ["i love deep learning", "deep learning is fun", "i love fun", "i love funny peoples","i love deep learning so much" ]

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
word2idx = tokenizer.word_index
vocab_size = len(word2idx) + 1   
print("Vocabulary:", word2idx)


Vocabulary: {'i': 1, 'love': 2, 'deep': 3, 'learning': 4, 'fun': 5, 'is': 6, 'funny': 7, 'peoples': 8, 'so': 9, 'much': 10}


In [4]:
window = 2
contexts = []
targets = []

for sentence in sentences:
    tokens = sentence.split()
    for idx, center_word in enumerate(tokens):
        start = max(idx - window, 0)
        end = min(idx + window, len(tokens) - 1)
        for i in range(start, end + 1):
            if i == idx:
                continue
            context_word = tokens[i]
            contexts.append(word2idx[context_word])  
            targets.append(word2idx[center_word])    

X = np.array(contexts)      
y = np.array(targets)      

X = X.reshape(-1, 1)         

Y = to_categorical(y, num_classes=vocab_size) 

print("X shape:", X.shape, "Y shape:", Y.shape)

X shape: (54, 1) Y shape: (54, 11)


In [5]:
embedding_dim = 5
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=1))
model.add(Lambda(lambda x: K.mean(x, axis=1)))   
model.add(Dense(vocab_size, activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

model.fit(X, Y, epochs=200, batch_size=8, verbose=0)

word = "deep"
idx = word2idx[word]
emb_matrix = model.get_weights()[0]
print(f"Embedding for '{word}' (index {idx}):\n", emb_matrix[idx])

C:\Users\swapn_mchak63\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Embedding for 'deep' (index 3):
 [-0.848519    0.00762022  0.17428753 -0.690895   -0.14888044]


In [6]:
model.fit(X, Y, epochs=200, batch_size=8, verbose=0)

In [7]:
word = "deep"
idx = word2idx[word]
emb_matrix = model.get_weights()[0]
print(f"Embedding for '{word}' (index {idx}):\n", emb_matrix[idx])

Embedding for 'deep' (index 3):
 [-1.2065823   0.0203681   0.2680539  -0.81746083 -0.18382852]


In [8]:
id2word = {v:k for k,v in word2idx.items()}
for i in range(min(10, X.shape[0])):
    context_id = X[i,0]
    target_id = y[i]
    print(f"Context -> Target: {id2word[context_id]}  ->  {id2word[target_id]}")

Context -> Target: love  ->  i
Context -> Target: deep  ->  i
Context -> Target: i  ->  love
Context -> Target: deep  ->  love
Context -> Target: learning  ->  love
Context -> Target: i  ->  deep
Context -> Target: love  ->  deep
Context -> Target: learning  ->  deep
Context -> Target: love  ->  learning
Context -> Target: deep  ->  learning


In [9]:
def predict_similar(word):
    idx = word2idx.get(word)
    if idx is None:
        print("word not in vocab")
        return
    
    x = np.array([[idx]])
    
    pred = model.predict(x, verbose=0)[0]   
    best = np.argmax(pred)
    print("Input word:", word)
    print("Most likely predicted word:", id2word[best])

In [10]:
predict_similar("deep")
predict_similar("learning")

Input word: deep
Most likely predicted word: learning
Input word: learning
Most likely predicted word: deep
